In [ ]:
import os 
os.environ['AWS_PROFILE'] = 'admin'
os.environ['HAVEN_DATABASE'] = 'haven'

import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
from random import choice
import h3

from mirrorverse.utils import read_data_w_cache
from mirrorverse.plotting import build_geojson

In [ ]:
sql = '''
select
    h3_index,
    n_salinity,
    habitat,
    elevation,
    extract(month from time) as month
from 
    chinook_depth_full_inference_3_5_1
where 
    extract(day from time) = 1
    and depth_bin = 25
'''
data = read_data_w_cache(sql)
data['lat'] = data['h3_index'].apply(lambda x: h3.h3_to_geo(x)[0])
data['lon'] = data['h3_index'].apply(lambda x: h3.h3_to_geo(x)[1])

from shapely.geometry import Polygon, Point

poly = Polygon(
    [
        (-166, 54.4),
        (-160, 56),
        (-158, 57.2),
        (-153, 62),
        (-149, 62),
        (-146, 62),
        (-140, 60),
        (-136, 58.4),
        (-133, 57.5),
        (-132, 56.0),
        (-131, 55),
        (-125, 50.3),
        (-170, 52.5),
        (-166, 54.4),
    ]
)
data['inside_polygon'] = data.apply(lambda row: poly.contains(Point(row['lon'], row['lat'])), axis=1)
data = data[data['inside_polygon']]
data = data[data['lon'] < -145]
data = data[data['elevation'] > -600]
data = data.drop_duplicates(['h3_index', 'month']).reset_index(drop=True)
print(data.shape)
data.head()

In [ ]:
plot_me(data, 'habitat')

In [ ]:
sql = '''
select
    _individual,
    _decision,
    _selected,
    extract(month from time) as month,
    extract(year from time) as year,
    h3_index,
    depth_bin,
    cos_sun,
    sin_sun,
    n_salinity,
    n_nitrate,
    habitat,
    probability
from 
    chinook_depth_inference_3_5_1
where 
    run_id = 'a43c72eff441211221256194305afd6a9ceb5a277a01f3853cf2716fa06725e8'
    and _train
'''
habitat_data = read_data_w_cache(sql)
print(habitat_data.shape)
habitat_data.head()

In [ ]:
sql = '''
select
    _individual,
    _decision,
    extract(month from time) as month,
    extract(year from time) as year,
    h3_index,
    depth_bin,
    cos_sun,
    sin_sun,
    first_value(n_salinity) over (partition by _individual, _decision order by depth_bin asc) as n_salinity,
    first_value(n_nitrate) over (partition by _individual, _decision order by depth_bin asc) as n_nitrate,
    probability
from 
    chinook_depth_inference_3_1_4
where 
    run_id = '00cf23b296999368ea18b82e33b8687c51e8c35e876afd325e26317cb69ea45b'
    and _selected
    and _train
'''
base_data = read_data_w_cache(sql)
print(base_data.shape)
base_data.head()

In [ ]:
data = habitat_data[habitat_data['_selected']].merge(
    base_data[['_individual', '_decision', 'probability']].rename(columns={'probability': 'base_probability'})
)
data['nll'] = np.log(data['probability'])
data['base_nll'] = np.log(data['base_probability'])
data['diff'] = data['nll'] - data['base_nll']
data.head()

In [ ]:
df = data.groupby(['month', 'h3_index'])[['diff', 'n_salinity', 'n_nitrate', 'habitat', 'nll', 'base_nll']].mean().reset_index()

In [ ]:
def plot_me(df, feature):
    fig = go.Figure()
    geojson = build_geojson(df, 'h3_index')
    months = sorted(df['month'].unique())
    for month in months:
        sdf = df[(df['month'] == month)]
        fig.add_trace(
            go.Choroplethmapbox(
                geojson=geojson,
                locations=sdf['h3_index'],
                z=sdf[feature],
                visible=False,
                marker_line_color='rgba(255,255,255,0)',
                #zmin=df[feature].min(),
                #zmax=df[feature].max(),
                #colorscale='algae'
            )
        )

    fig.data[0].visible = True

    steps = []
    for i, slider_val in enumerate(months):
        step = dict(
            method="update",
            args=[
                {"visible": [False] * len(months)},
                {"title": f"month: {slider_val}"},
            ],
            label=f"{slider_val}"
        )
        step["args"][0]["visible"][i] = True
        steps.append(step)

    sliders = [dict(
        active=0,
        currentvalue={"prefix": f"feature: "},
        pad={"t": 50, "b": 25, "l": 25},
        steps=steps
    )]

    fig.update_layout(
        sliders=sliders
    )

    fig.update_layout(
        autosize=False,  # Disable autosizing
        width=800,       # Set width in pixels
        height=800,      # Set height in pixels
    )

    fig.update_layout(
        margin={"r":0,"t":30,"l":0,"b":0}, mapbox=dict(style="carto-positron", zoom=3, center = {"lat": 57, "lon": -150})
    )
    return fig

In [ ]:
plot_me(df, 'nll').show()

In [ ]:
plot_me(df, 'habitat').show()

In [ ]:
plot_me(df, 'n_salinity').show()

In [ ]:
px.scatter(df, x='n_salinity', y='base_nll')

In [ ]:
salinity = (
    habitat_data[habitat_data['depth_bin'] == 25].groupby(['_individual', '_decision'])
    [['n_salinity', 'habitat']].mean().reset_index()
)
nll = base_data.copy()
nll['nll'] = np.log(nll['probability'])
df = salinity.merge(nll[['_individual', '_decision', 'nll', 'month', 'h3_index']])
df = df.groupby(['month', 'h3_index'])[['nll', 'n_salinity', 'habitat']].mean().reset_index()
px.scatter(df, x='n_salinity', y='nll', opacity=0.5)

In [ ]:
px.scatter(df, x='habitat', y='nll', opacity=0.5)

In [ ]:
salinity = (
    habitat_data[habitat_data['depth_bin'] == 25].groupby(['_individual', '_decision'])
    [['n_salinity', 'habitat']].mean().reset_index()
)
nll = base_data.copy()
nll['nll'] = np.log(nll['probability'])
df = salinity.merge(nll[['_individual', '_decision', 'nll', 'month', 'h3_index']])
df['winter'] = df['month'].isin([1, 2, 3])
df['qcut'] = pd.qcut(df['n_salinity'], q=5)
df = df.groupby(['winter', 'qcut'])[['n_salinity', 'nll']].mean().reset_index()
px.scatter(df, x='n_salinity', y='nll', color='winter')

In [ ]:
salinity = (
    habitat_data[habitat_data['depth_bin'] == 25].groupby(['_individual', '_decision'])
    [['n_salinity', 'habitat']].mean().reset_index()
)
nll = base_data.copy()
nll['nll'] = np.log(nll['probability'])
df = salinity.merge(nll[['_individual', '_decision', 'nll', 'month', 'h3_index']])
df = df.groupby(pd.qcut(df['habitat'], q=5))[['habitat', 'nll']].mean()
px.scatter(df, x='habitat', y='nll')